In [ ]:
import json
import warnings
from pathlib import Path
from typing import Any, Dict, Iterable, Optional, Sequence, Tuple

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from matplotlib.colors import LinearSegmentedColormap, Normalize
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import ParameterGrid, train_test_split

warnings.filterwarnings("default")
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

RANDOM_STATE = 42
TEST_SIZE = 1.0 / 3.0
VALIDATION_FRACTION_OF_REMAINDER = 0.5
EARLY_STOPPING_ROUNDS = 30
CLASSIFICATION_THRESHOLD = 0.5
TOP_N_GENES = 10
PLOT_DPI = 300

LTS_LABEL = 0
HTS_LABEL = 1
CLASS_NAMES = ["LTS", "HTS"]

LTS_COLOR = "#80bcc8"
HTS_COLOR = "#d88f91"

SHAP_CMAP = LinearSegmentedColormap.from_list(
    "custom_shap_importance",
    ["#39489f", "#39bbec", "#f9ed36", "#f38466", "#b81f25"],
    N=256,
)

plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = PLOT_DPI
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11


def get_code_dir() -> Path:
    """Return the script directory, or the current directory in a notebook."""
    try:
        return Path(__file__).resolve().parent
    except NameError:
        return Path.cwd()


def ensure_dir(path: Path) -> None:
    """Create a directory and its parents when they do not already exist."""
    path.mkdir(parents=True, exist_ok=True)


def save_figure(fig: plt.Figure, save_path: Path) -> None:
    """Save and close a Matplotlib figure."""
    fig.savefig(
        save_path,
        dpi=PLOT_DPI,
        bbox_inches="tight",
        facecolor="white",
        transparent=False,
    )
    plt.close(fig)
    print(f"[SAVED] {save_path}")


def save_confusion_matrix(
    cm: np.ndarray,
    class_names: Sequence[str],
    save_path: Path,
    title: str = "CatBoost Test Confusion Matrix",
) -> None:
    """Save a confusion-matrix heatmap."""
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    image = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    fig.colorbar(image, ax=ax)

    ax.set(
        xticks=np.arange(cm.shape[1]),
        yticks=np.arange(cm.shape[0]),
        xticklabels=class_names,
        yticklabels=class_names,
        ylabel="True label",
        xlabel="Predicted label",
        title=title,
    )

    threshold = cm.max() / 2.0 if cm.max() > 0 else 0.5
    for row_idx in range(cm.shape[0]):
        for col_idx in range(cm.shape[1]):
            ax.text(
                col_idx,
                row_idx,
                format(cm[row_idx, col_idx], "d"),
                ha="center",
                va="center",
                color="white" if cm[row_idx, col_idx] > threshold else "black",
            )

    fig.tight_layout()
    save_figure(fig, save_path)


def plot_roc_curve(y_true: np.ndarray, y_prob: np.ndarray, save_path: Path) -> None:
    """Save the receiver operating characteristic curve."""
    auc_value = roc_auc_score(y_true, y_prob)
    fpr, tpr, _ = roc_curve(y_true, y_prob)

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.plot(fpr, tpr, linewidth=2, label=f"ROC AUC = {auc_value:.4f}")
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1.5)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("CatBoost Test ROC Curve")
    ax.legend(loc="lower right")

    fig.tight_layout()
    save_figure(fig, save_path)


def plot_pr_curve(y_true: np.ndarray, y_prob: np.ndarray, save_path: Path) -> None:
    """Save the precision-recall curve."""
    average_precision = average_precision_score(y_true, y_prob)
    precision, recall, _ = precision_recall_curve(y_true, y_prob)

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.plot(recall, precision, linewidth=2, label=f"AP = {average_precision:.4f}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("CatBoost Test Precision-Recall Curve")
    ax.legend(loc="lower left")

    fig.tight_layout()
    save_figure(fig, save_path)


def plot_test_score_distribution(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    save_path: Path,
) -> None:
    """Save the distribution of predicted HTS probabilities in the test set."""
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.hist(
        y_prob[y_true == LTS_LABEL],
        bins=30,
        alpha=0.7,
        label="LTS (0)",
        color=LTS_COLOR,
        edgecolor="black",
    )
    ax.hist(
        y_prob[y_true == HTS_LABEL],
        bins=30,
        alpha=0.7,
        label="HTS (1)",
        color=HTS_COLOR,
        edgecolor="black",
    )
    ax.set_xlabel("Predicted probability of HTS")
    ax.set_ylabel("Cell count")
    ax.set_title("CatBoost Test Prediction Score Distribution")
    ax.legend()

    fig.tight_layout()
    save_figure(fig, save_path)


def plot_shap_importance_bar(
    shap_importance_df: pd.DataFrame,
    save_path: Path,
    top_n: int = TOP_N_GENES,
) -> None:
    """Plot the highest-ranking genes by mean absolute SHAP value."""
    plot_df = (
        shap_importance_df
        .sort_values("mean_abs_shap", ascending=False)
        .head(top_n)
        .iloc[::-1]
        .reset_index(drop=True)
    )

    if plot_df.empty:
        print("[WARNING] No SHAP importance values were available for plotting.")
        return

    genes = plot_df["gene"].astype(str).to_numpy()
    values = plot_df["mean_abs_shap"].astype(float).to_numpy()

    value_min = float(values.min())
    value_max = float(values.max())
    if np.isclose(value_min, value_max):
        norm = Normalize(vmin=0.0, vmax=max(value_max, 1.0))
    else:
        norm = Normalize(vmin=value_min, vmax=value_max)

    bar_colors = SHAP_CMAP(norm(values))
    fig_height = max(5.0, 0.42 * len(plot_df))
    fig, ax = plt.subplots(figsize=(7.0, fig_height))

    y_position = np.arange(len(plot_df))
    ax.barh(
        y_position,
        values,
        color=bar_colors,
        edgecolor="none",
        height=0.82,
    )
    ax.set_yticks(y_position)
    ax.set_yticklabels(genes)
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_ylabel("Gene")
    ax.set_title(f"Top {len(plot_df)} CatBoost Genes by SHAP Importance")

    scalar_mappable = mpl.cm.ScalarMappable(cmap=SHAP_CMAP, norm=norm)
    scalar_mappable.set_array([])
    colorbar = fig.colorbar(scalar_mappable, ax=ax, fraction=0.08, pad=0.08)
    colorbar.set_label("Mean |SHAP value|")

    fig.tight_layout()
    save_figure(fig, save_path)


def plot_class_specific_shap_bar(
    shap_importance_df: pd.DataFrame,
    save_path: Path,
    top_n: int = TOP_N_GENES,
) -> None:
    """Compare class-specific mean absolute SHAP values for top genes."""
    plot_df = (
        shap_importance_df
        .sort_values("mean_abs_shap", ascending=False)
        .head(top_n)
        .iloc[::-1]
        .reset_index(drop=True)
    )

    if plot_df.empty:
        print("[WARNING] No class-specific SHAP values were available for plotting.")
        return

    y_position = np.arange(len(plot_df))
    bar_height = 0.38

    fig_height = max(5.0, 0.42 * len(plot_df))
    fig, ax = plt.subplots(figsize=(7.0, fig_height))
    ax.barh(
        y_position - bar_height / 2.0,
        plot_df["mean_abs_shap_LTS"],
        height=bar_height,
        label="LTS (0)",
        color=LTS_COLOR,
    )
    ax.barh(
        y_position + bar_height / 2.0,
        plot_df["mean_abs_shap_HTS"],
        height=bar_height,
        label="HTS (1)",
        color=HTS_COLOR,
    )
    ax.set_yticks(y_position)
    ax.set_yticklabels(plot_df["gene"].astype(str))
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_ylabel("Gene")
    ax.set_title(f"Top {len(plot_df)} CatBoost Genes by Class-Specific SHAP Importance")
    ax.legend()

    fig.tight_layout()
    save_figure(fig, save_path)


def extract_binary_shap_values(shap_values: Any, n_features: int) -> np.ndarray:
    """Normalize common binary-classification SHAP output formats."""
    if hasattr(shap_values, "values"):
        shap_values = shap_values.values

    if isinstance(shap_values, list):
        if len(shap_values) == 2:
            shap_values = shap_values[1]
        elif len(shap_values) == 1:
            shap_values = shap_values[0]
        else:
            shap_values = np.asarray(shap_values)

    values = np.asarray(shap_values)

    if values.ndim == 2:
        if values.shape[1] == n_features:
            return values
        if values.shape[1] == n_features + 1:
            return values[:, :n_features]

    if values.ndim == 3:
        if values.shape[1] == 2 and values.shape[2] == n_features:
            return values[:, 1, :]
        if values.shape[1] == n_features and values.shape[2] == 2:
            return values[:, :, 1]

    raise ValueError(f"Unexpected SHAP output shape: {values.shape}")


def build_catboost_model(
    params: Dict[str, Any],
    verbose: int = 0,
) -> CatBoostClassifier:
    """Create a binary CatBoost classifier with reproducible settings."""
    return CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=RANDOM_STATE,
        verbose=verbose,
        allow_writing_files=False,
        **params,
    )


def get_effective_iterations(
    model: CatBoostClassifier,
    default_iterations: int,
) -> int:
    """Return the number of boosting iterations selected by early stopping."""
    best_iteration = model.get_best_iteration()
    if best_iteration is None or best_iteration < 0:
        return int(default_iterations)
    return int(best_iteration) + 1


def identify_feature_columns(df: pd.DataFrame) -> Tuple[Sequence[str], Sequence[str]]:
    """Identify numeric feature columns and excluded metadata columns."""
    excluded_columns = ["label"]

    if df.columns[0] != "label":
        first_column = df.columns[0]
        if not pd.api.types.is_numeric_dtype(df[first_column]):
            excluded_columns.append(first_column)

    feature_columns = [column for column in df.columns if column not in excluded_columns]
    return feature_columns, excluded_columns


def load_dataset(data_path: Path) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, Sequence[str]]:
    """Read and validate the expression matrix and binary labels."""
    if not data_path.exists():
        raise FileNotFoundError(f"Data file not found: {data_path}")

    df = pd.read_csv(data_path)
    if "label" not in df.columns:
        raise ValueError("The dataset must contain a 'label' column.")

    feature_columns, excluded_columns = identify_feature_columns(df)
    if not feature_columns:
        raise ValueError("No feature columns were found in the dataset.")

    numeric_features = df[feature_columns].apply(pd.to_numeric, errors="coerce")
    missing_count = int(numeric_features.isna().sum().sum())
    if missing_count > 0:
        print(
            f"[WARNING] {missing_count} feature values were missing or non-numeric "
            "and were replaced with 0."
        )

    X = numeric_features.fillna(0.0).astype(np.float32)
    y = pd.to_numeric(df["label"], errors="coerce")

    if y.isna().any() or not set(y.dropna().unique()).issubset({LTS_LABEL, HTS_LABEL}):
        raise ValueError("The 'label' column must contain only 0 (LTS) and 1 (HTS).")

    y = y.astype(int)

    print(f"[INFO] Excluded non-feature columns: {excluded_columns}")
    print(f"[INFO] Dataset shape: {df.shape}")
    print(f"[INFO] Feature matrix shape: {X.shape}")
    print(f"[INFO] LTS (0): {int((y == LTS_LABEL).sum())}")
    print(f"[INFO] HTS (1): {int((y == HTS_LABEL).sum())}")

    return df, X, y, feature_columns


def tune_catboost(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_val: pd.DataFrame,
    y_val: pd.Series,
    output_dir: Path,
) -> Tuple[Dict[str, Any], int, float]:
    """Select CatBoost hyperparameters by validation-set ROC AUC."""
    parameter_grid = {
        "iterations": [200, 400],
        "depth": [4, 6, 8],
        "learning_rate": [0.03, 0.1],
        "l2_leaf_reg": [3, 5],
    }

    best_params: Optional[Dict[str, Any]] = None
    best_iterations: Optional[int] = None
    best_validation_auc = -np.inf
    tuning_records = []

    grid = list(ParameterGrid(parameter_grid))
    print(f"[INFO] Evaluating {len(grid)} CatBoost parameter combinations...")

    for trial_index, params in enumerate(grid, start=1):
        model = build_catboost_model(params, verbose=0)
        model.fit(
            X_train,
            y_train,
            eval_set=(X_val, y_val),
            use_best_model=True,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            verbose=False,
        )

        validation_prob = model.predict_proba(X_val)[:, HTS_LABEL]
        validation_pred = (validation_prob >= CLASSIFICATION_THRESHOLD).astype(int)

        validation_auc = roc_auc_score(y_val, validation_prob)
        validation_accuracy = accuracy_score(y_val, validation_pred)
        validation_f1 = f1_score(y_val, validation_pred, zero_division=0)
        validation_precision = precision_score(y_val, validation_pred, zero_division=0)
        validation_recall = recall_score(y_val, validation_pred, zero_division=0)
        effective_iterations = get_effective_iterations(model, params["iterations"])

        tuning_records.append(
            {
                "trial": trial_index,
                "iterations_max": params["iterations"],
                "iterations_used": effective_iterations,
                "depth": params["depth"],
                "learning_rate": params["learning_rate"],
                "l2_leaf_reg": params["l2_leaf_reg"],
                "val_auc": validation_auc,
                "val_accuracy": validation_accuracy,
                "val_f1": validation_f1,
                "val_precision": validation_precision,
                "val_recall": validation_recall,
            }
        )

        print(
            f"[INFO] Trial {trial_index}/{len(grid)} | "
            f"iterations_max={params['iterations']} | "
            f"iterations_used={effective_iterations} | "
            f"depth={params['depth']} | "
            f"learning_rate={params['learning_rate']} | "
            f"l2_leaf_reg={params['l2_leaf_reg']} | "
            f"Val AUC={validation_auc:.4f}"
        )

        if validation_auc > best_validation_auc:
            best_validation_auc = validation_auc
            best_params = params.copy()
            best_iterations = effective_iterations

    if best_params is None or best_iterations is None:
        raise RuntimeError("No CatBoost model was trained successfully.")

    tuning_df = pd.DataFrame(tuning_records).sort_values("val_auc", ascending=False)
    tuning_df.to_csv(output_dir / "validation_tuning_results.csv", index=False)

    selected_params = best_params.copy()
    selected_params["iterations"] = int(best_iterations)

    with open(output_dir / "best_catboost_parameters.json", "w", encoding="utf-8") as handle:
        json.dump(
            {
                "best_parameters": selected_params,
                "best_validation_auc": best_validation_auc,
                "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
            },
            handle,
            indent=4,
        )

    print(f"[INFO] Best parameters: {selected_params}")
    print(f"[INFO] Best validation AUC: {best_validation_auc:.4f}")

    return selected_params, int(best_iterations), float(best_validation_auc)


def calculate_final_model_shap(
    final_model: CatBoostClassifier,
    X_all: pd.DataFrame,
    y_all: pd.Series,
    feature_columns: Sequence[str],
    output_dir: Path,
    figure_dir: Path,
) -> None:
    """Calculate SHAP values for the final model over the full dataset."""
    try:
        import shap

        print("[INFO] Calculating SHAP values for the final CatBoost model...")
        explainer = shap.TreeExplainer(final_model)
        raw_shap_values = explainer.shap_values(X_all)
        shap_values = extract_binary_shap_values(
            raw_shap_values,
            n_features=len(feature_columns),
        )

        if shap_values.shape != X_all.shape:
            raise ValueError(
                "SHAP dimensions do not match the feature matrix: "
                f"SHAP={shap_values.shape}, X={X_all.shape}"
            )

        y_array = np.asarray(y_all)
        lts_mask = y_array == LTS_LABEL
        hts_mask = y_array == HTS_LABEL

        mean_abs_all = np.abs(shap_values).mean(axis=0)
        mean_abs_lts = np.abs(shap_values[lts_mask]).mean(axis=0)
        mean_abs_hts = np.abs(shap_values[hts_mask]).mean(axis=0)
        mean_shap = shap_values.mean(axis=0)

        shap_importance_df = pd.DataFrame(
            {
                "gene": feature_columns,
                "mean_abs_shap": mean_abs_all,
                "mean_abs_shap_LTS": mean_abs_lts,
                "mean_abs_shap_HTS": mean_abs_hts,
                "mean_shap": mean_shap,
            }
        ).sort_values("mean_abs_shap", ascending=False)

        shap_importance_df.to_csv(
            output_dir / "shap_gene_importance_final_catboost.csv",
            index=False,
        )
        shap_importance_df.head(TOP_N_GENES).to_csv(
            output_dir / f"top{TOP_N_GENES}_shap_genes_final_catboost.csv",
            index=False,
        )

        shap.summary_plot(
            shap_values,
            X_all,
            feature_names=feature_columns,
            show=False,
            plot_type="dot",
            max_display=30,
        )
        summary_figure = plt.gcf()
        summary_figure.set_size_inches(10, 8)
        save_figure(
            summary_figure,
            figure_dir / "shap_summary_beeswarm_final_catboost.svg",
        )

        plot_shap_importance_bar(
            shap_importance_df,
            figure_dir / f"top{TOP_N_GENES}_shap_importance_final_catboost.svg",
            top_n=TOP_N_GENES,
        )
        plot_class_specific_shap_bar(
            shap_importance_df,
            figure_dir / f"top{TOP_N_GENES}_class_specific_shap_final_catboost.svg",
            top_n=TOP_N_GENES,
        )

    except Exception as error:
        raise RuntimeError(f"CatBoost SHAP analysis failed: {error}") from error


def main() -> None:
    code_dir = get_code_dir()
    data_path = code_dir / "df_expr.csv"
    output_dir = code_dir / "catboost_hts_lts_results"
    figure_dir = output_dir / "figures"

    ensure_dir(output_dir)
    ensure_dir(figure_dir)

    print(f"[INFO] Working directory: {code_dir}")
    print(f"[INFO] Reading dataset: {data_path}")

    _, X, y, feature_columns = load_dataset(data_path)

    X_remainder, X_test, y_remainder, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_remainder,
        y_remainder,
        test_size=VALIDATION_FRACTION_OF_REMAINDER,
        random_state=RANDOM_STATE,
        stratify=y_remainder,
    )

    print(f"[INFO] Training set size: {X_train.shape[0]}")
    print(f"[INFO] Validation set size: {X_val.shape[0]}")
    print(f"[INFO] Test set size: {X_test.shape[0]}")

    best_params, best_iterations, best_validation_auc = tune_catboost(
        X_train,
        y_train,
        X_val,
        y_val,
        output_dir,
    )

    X_trainval = pd.concat([X_train, X_val], axis=0)
    y_trainval = pd.concat([y_train, y_val], axis=0)

    final_params = best_params.copy()
    final_params["iterations"] = best_iterations
    final_model = build_catboost_model(final_params, verbose=0)

    print("[INFO] Retraining the final CatBoost model on training and validation data...")
    final_model.fit(X_trainval, y_trainval, verbose=False)

    feature_importance_df = pd.DataFrame(
        {
            "gene": feature_columns,
            "catboost_feature_importance": final_model.get_feature_importance(),
        }
    ).sort_values("catboost_feature_importance", ascending=False)
    feature_importance_df.to_csv(
        output_dir / "catboost_feature_importance_final_model.csv",
        index=False,
    )

    print("[INFO] Evaluating the final CatBoost model on the independent test set...")
    test_probability = final_model.predict_proba(X_test)[:, HTS_LABEL]
    test_prediction = (test_probability >= CLASSIFICATION_THRESHOLD).astype(int)

    test_auc = roc_auc_score(y_test, test_probability)
    test_accuracy = accuracy_score(y_test, test_prediction)
    test_f1 = f1_score(y_test, test_prediction, zero_division=0)
    test_precision = precision_score(y_test, test_prediction, zero_division=0)
    test_recall = recall_score(y_test, test_prediction, zero_division=0)
    test_average_precision = average_precision_score(y_test, test_probability)

    metrics_df = pd.DataFrame(
        [
            {
                "best_validation_auc": best_validation_auc,
                "test_auc": test_auc,
                "test_accuracy": test_accuracy,
                "test_f1": test_f1,
                "test_precision": test_precision,
                "test_recall": test_recall,
                "test_average_precision": test_average_precision,
                "classification_threshold": CLASSIFICATION_THRESHOLD,
                "best_iterations": best_iterations,
                "best_depth": best_params["depth"],
                "best_learning_rate": best_params["learning_rate"],
                "best_l2_leaf_reg": best_params["l2_leaf_reg"],
                "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
            }
        ]
    )
    metrics_df.to_csv(output_dir / "test_metrics.csv", index=False)

    report = classification_report(
        y_test,
        test_prediction,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )
    with open(output_dir / "classification_report.txt", "w", encoding="utf-8") as handle:
        handle.write(report)

    test_prediction_df = pd.DataFrame(
        {
            "true_label": y_test.to_numpy(),
            "true_class": y_test.map({LTS_LABEL: "LTS", HTS_LABEL: "HTS"}).to_numpy(),
            "predicted_label": test_prediction,
            "predicted_class": pd.Series(test_prediction).map(
                {LTS_LABEL: "LTS", HTS_LABEL: "HTS"}
            ).to_numpy(),
            "predicted_probability_HTS": test_probability,
        }
    )
    test_prediction_df.to_csv(output_dir / "test_predictions.csv", index=False)

    test_confusion_matrix = confusion_matrix(y_test, test_prediction)
    save_confusion_matrix(
        test_confusion_matrix,
        CLASS_NAMES,
        figure_dir / "test_confusion_matrix.svg",
    )
    plot_roc_curve(
        y_test.to_numpy(),
        test_probability,
        figure_dir / "test_roc_curve.svg",
    )
    plot_pr_curve(
        y_test.to_numpy(),
        test_probability,
        figure_dir / "test_precision_recall_curve.svg",
    )
    plot_test_score_distribution(
        y_test.to_numpy(),
        test_probability,
        figure_dir / "test_prediction_score_distribution.svg",
    )

    calculate_final_model_shap(
        final_model,
        X,
        y,
        feature_columns,
        output_dir,
        figure_dir,
    )

    print("[RESULT] Analysis completed successfully.")
    print(f"[RESULT] Output directory: {output_dir}")
    print(f"[RESULT] Best validation AUC: {best_validation_auc:.4f}")
    print(f"[RESULT] Test AUC: {test_auc:.4f}")
    print(f"[RESULT] Test accuracy: {test_accuracy:.4f}")
    print(f"[RESULT] Test F1 score: {test_f1:.4f}")
    print(f"[RESULT] Test precision: {test_precision:.4f}")
    print(f"[RESULT] Test recall: {test_recall:.4f}")
    print(f"[RESULT] Test average precision: {test_average_precision:.4f}")


if __name__ == "__main__":
    main()
